# CNN facial-expression results

This notebook reports the trained convolutional model used by the Tkinter app. It reads saved experiment artifacts and does not retrain or download the dataset.

## Why convolutional layers fit face images

Each input is a 48×48 grayscale face. The CNN keeps the row and column axes, then learns 3×3 filters across three stages (16, 32, and 64 feature maps). Each stage applies batch normalization, ReLU, max pooling, and dropout. Adaptive pooling feeds a 128-unit dense layer and seven expression outputs. Random horizontal flips augment only training images; inference averages probabilities for the original and mirrored crop.

In [1]:
from pathlib import Path
import json
import pandas as pd

project = next((parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'models/selected_model.json').is_file()), None)
if project is None:
    raise FileNotFoundError('Open this notebook inside the facial-emotion-recognition project folder.')
run = project / 'results/cnn_ferplus'
validation = json.loads((run / 'validation_metrics.json').read_text(encoding='utf-8'))
test = json.loads((run / 'final_test_metrics_ferplus_consensus.json').read_text(encoding='utf-8'))
history = json.loads((run / 'training_history.json').read_text(encoding='utf-8'))
metadata = json.loads((project / 'models/selected_model.json').read_text(encoding='utf-8'))
print(f"Run artifacts: {run.relative_to(project)}")
print(metadata['architecture'])
print(f"Training: {len(history)} epochs, {validation['training_seconds']:.1f} seconds on {validation['device']}")

Run: .\results\cnn_ferplus
3 convolution blocks (16,32,64), 3x3 conv per block, batch normalization, ReLU, max pooling, adaptive 3x3 pooling, 128-unit dense head; 48x48 grayscale input; train-only random horizontal flips
Training: 18 epochs, 955.4 seconds on cpu


## Validation comparison

Both models were scored on the same 3,356 accepted FER+ PublicTest consensus rows. The prior MLP score is included as the selection baseline; the PrivateTest set remains separate until after candidate selection.

In [2]:
validation_table = pd.DataFrame([
    {'Model': 'FER+ MLP baseline', 'Accuracy': validation['baseline_current_model']['accuracy'], 'Macro-F1': validation['baseline_current_model']['macro_f1']},
    {'Model': 'FER+ spatial CNN + flip averaging', 'Accuracy': validation['cnn_candidate']['accuracy'], 'Macro-F1': validation['cnn_candidate']['macro_f1']},
]).set_index('Model')
validation_table.style.format('{:.4f}')

,Accuracy,Macro-F1
Model,,
FER+ MLP baseline,0.5498,0.3636
FER+ spatial CNN + flip averaging,0.7378,0.5159


## Held-out PrivateTest result

The selected CNN was evaluated once on 3,329 accepted consensus rows. These FER+ metrics use a filtered consensus-label subset and are not directly comparable to runs scored against the original FER2013 labels.

In [3]:
test_table = pd.DataFrame([{'Split': 'PrivateTest (FER+ consensus)', 'Samples': int(test['classification_report']['weighted avg']['support']), 'Accuracy': test['accuracy'], 'Macro-F1': test['macro_f1']}]).set_index('Split')
display(test_table.style.format({'Accuracy': '{:.4f}', 'Macro-F1': '{:.4f}', 'Samples': '{:,.0f}'}))

per_class = pd.DataFrame(test['classification_report']).T.loc[metadata['class_names'], ['precision', 'recall', 'f1-score', 'support']]
per_class.index.name = 'Expression'
display(per_class.style.format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1-score': '{:.3f}', 'support': '{:,.0f}'}))

,Samples,Accuracy,Macro-F1
Split,,,
PrivateTest (FER+ consensus),"3,329",0.7188,0.5081


,precision,recall,f1-score,support
Expression,,,,
Angry,0.675,0.529,0.593,310
Disgust,0.000,0.000,0.000,22
Fear,0.800,0.133,0.229,90
Happy,0.817,0.834,0.826,905
Sad,0.531,0.285,0.371,421
Surprise,0.731,0.825,0.775,422
Neutral,0.688,0.858,0.763,"1,159"


## Training history

Early stopping retained the best validation-loss checkpoint. Per-epoch losses and single-view validation accuracy are shown below. The reported model metrics above include horizontal-flip probability averaging.

In [4]:
history_table = pd.DataFrame(history).set_index('epoch')
display(history_table.style.format({'train_loss': '{:.4f}', 'validation_loss': '{:.4f}', 'validation_accuracy_single_view': '{:.4f}', 'learning_rate': '{:.5f}'}))
print('Final model metadata:', {k: metadata[k] for k in ('name', 'inference_strategy', 'validation_accuracy', 'validation_macro_f1', 'run')})

,train_loss,validation_loss,validation_accuracy_single_view,learning_rate
epoch,,,,
1,1.5181,1.3648,0.4949,0.00080
2,1.3136,1.1700,0.5888,0.00080
3,1.1926,1.0745,0.6192,0.00080
4,1.1208,1.0083,0.6505,0.00080
5,1.0669,0.9724,0.6538,0.00080
6,1.0279,0.9211,0.6788,0.00080
7,0.9927,0.8889,0.6800,0.00080
8,0.9646,0.8715,0.6877,0.00080
9,0.9348,0.8431,0.6973,0.00080


Final model metadata: {'name': 'ferplus_spatial_cnn_16_32_64', 'inference_strategy': 'horizontal_flip_probability_average', 'validation_accuracy': 0.7377830750893921, 'validation_macro_f1': 0.5158889563053426, 'run': 'results\\runs\\20261122T000557Z_cnn_ferplus'}
